In [16]:
#import libraries

import groq
import langchain
import pinecone
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings 

In [25]:
from dotenv import load_dotenv 
load_dotenv()

True

In [18]:
import os

In [19]:
def read_doc(directory):
    file_loader = DirectoryLoader(
        directory,
        glob="**/*.pdf", 
        loader_cls=PyMuPDFLoader,
    )
    documents=file_loader.load()
    return documents

In [20]:
doc=read_doc('documents')

In [21]:
len(doc)

164

In [22]:
# Divide the docs into chunks

def chunk_data(docs, chunk_size=1000, chunk_overlap=150):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunked_docs=text_splitter.split_documents(docs)
    return chunked_docs

In [23]:
documents = chunk_data(docs=doc)
print(f"Number of chunks: {len(documents)}")

Number of chunks: 571


In [26]:
## embedding technique
embeddings = HuggingFaceEmbeddings(model_name='BAAI/bge-large-en-v1.5')

vectors = embeddings.embed_query("What is the last rating?")
print(f"Vector length: {len(vectors)}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9402.92it/s]


Vector length: 1024


In [27]:
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

# SECURITY FIX: Pulling the key securely
# Make sure to set os.environ["PINECONE_API_KEY"] = "your_new_key" before running
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
index_name = "langchainvector"

index = pc.Index(index_name)

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [28]:
## add documents to vectorstore
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(documents))]
vector_store.add_documents(documents=documents, id=uuids)

['a1e0ab8d-195c-4ff6-9dc0-a27b12a3964a',
 '454aeb48-f0b2-43ef-8dae-494d3b1d7b2c',
 '687c9075-b4c4-423d-ae82-9987c5097004',
 'd8ee123b-9abd-4fe1-8d6a-d4d81b1c5602',
 'efa11a5d-b0cb-4520-9373-5db32a39da29',
 '36665548-f805-48fa-a66b-d57c05535ec2',
 '3f80ca19-2aa3-473c-813c-37c52613a9f1',
 '0c3fb95d-d2a1-4a2b-944e-c39de41becb9',
 '8fffb4bc-bfde-43d5-a0f1-7d588f17f637',
 '9b227456-10b6-404f-ab4e-613707e40074',
 'c43a4717-dba3-4086-a7c4-3cd87543bc88',
 'c80d882c-c324-47be-a547-d514136c917c',
 '70f81975-15cc-4f95-a8a1-c5e2fd924fe3',
 'd679595c-df5f-4312-a0f7-fb11ebbb1f7b',
 '9f594be6-3049-43ac-8bd4-6f173093d859',
 'fd7e821b-a0b0-4970-999b-d4a4d502b403',
 '40f49e97-e6d9-4612-b8da-3fda35780dc1',
 'ea44a205-8bf0-46a6-a8d9-409031386b26',
 'c797506e-b19b-4baf-8be3-96c21bf11150',
 'fc5ef5a4-bd1e-4ba3-b0b9-ded3c2416997',
 'd5072f05-8a8b-4314-8454-8d45c6bc77b9',
 '7b85469c-cc14-4982-be4e-148e6ae413d0',
 '833c4038-7f60-46d9-86c0-0352346de69e',
 'c43d3ddb-82ff-4e86-908a-aa3f4a224508',
 'd2802962-5556-

In [29]:
## cosine similarity retrieve results from vectordb
def retrieve_query(query, k=3):
    matching_results=vector_store.similarity_search(query=query, k=k)
    return matching_results

In [30]:
# 1. Import the modern chain modules
from langchain_groq import ChatGroq
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
import os

# 2. Initialize your LLM
# Ensure you have os.environ["GROQ_API_KEY"] set securely!
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# 3. Create a Prompt Template
# This dictates how the LLM should behave and injects the retrieved text into the {context} variable.
system_prompt = (
    "You are an expert financial analyst assistant. Use the provided pieces of retrieved context "
    "to answer the user's question with absolute precision.\n\n"
    "CRITICAL RULES:\n"
    "1. Pay close attention to document dates (e.g., 2024, 2025, 2026). Always state which report year you are pulling data from in your final answer.\n"
    "2. If values change across chunks, prioritize the most recent year's data unless asked otherwise.\n"
    "3. Keep formatting clean using bullet points for metrics.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 4. Convert your Pinecone Vector Store into a Retriever
# We are grabbing the top 3 most relevant chunks based on your earlier logic
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [31]:
# 5. Build the Chains
# The document chain formats the retrieved documents to pass to the LLM
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# The retrieval chain connects the Pinecone retriever to the document chain
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [33]:
# 6. Test the Chain!
query = "What was the performance of our company in 2025"
response = rag_chain.invoke({"input": query})

print(response["answer"])

Unfortunately, I do not have any information about the company's performance in 2025 as it was not provided in the given context. The context only mentions the company's performance in FY2019 and 9M FY2020, but does not provide any information about the company's performance in 2025.

However, I can provide some general information about the company's strategy and performance in the past, as mentioned in the context:

**FY2019:**

* Revenue growth: 45% YoY
* Operating profits growth: 31% YoY
* Debt protection metrics: Healthy
* Leverage levels: Moderate
* Financial flexibility: Exceptionally high

**9M FY2020:**

* Revenue growth: 7% YoY
* Operating profits growth: 4% YoY
* Debt: The company's debt was expected to be impacted by the headwinds in the refining and petrochemicals business environment.

Please note that the context does not provide any information about the company's performance in 2025, and any information about the company's performance in 2025 would require additional d

In [86]:
## Keep Index But delete all the vectors

import os
from pinecone import Pinecone

# Connect to Pinecone
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
index = pc.Index("langchainvector")

# Delete all vectors in the default namespace
index.delete(delete_all=True)

print("Success: All vectors have been deleted. The index is now empty.")

Success: All vectors have been deleted. The index is now empty.


In [87]:
## Delete whole index with vectors
## start from fresh

import os
import time
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
index_name = "langchainvector"

# 1. Delete the existing index
if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)
    print(f"Deleted old '{index_name}' index.")
    
    # Wait a moment for Pinecone's backend to register the deletion
    time.sleep(5) 

Deleted old 'langchainvector' index.
